In [ ]:
import pandas as pd
from config import DATABASE_URL
from sqlalchemy import create_engine, text

ENGINE = create_engine(DATABASE_URL)

res = 4

import_index = [
    'CellID', 
    'Latitude', 
    'Longitude',
    'RSLatitude', 
    'RSLongitude', 
    'RSDistance', 
    'RSCellID', 
    'ValidSnap',
    'neighbors', 
    'ValidNeighbors'
    ]

df = pd.read_csv(f"datasets/initial_res_{res}_points.csv", usecols=import_index)

df



,CellID,Latitude,Longitude,RSLatitude,RSLongitude,RSDistance,RSCellID,ValidSnap,neighbors,ValidNeighbors
0,8412c87ffffffff,49.187436,-113.638778,49.047197,-113.687518,15992,8412c87ffffffff,True,"['8412c81ffffffff', '8412c85ffffffff', '8412ca...","['8412cabffffffff', '8412cbdffffffff', '8412cb..."
1,8412c91ffffffff,49.066718,-112.048207,49.074821,-112.022018,2110,8412c91ffffffff,True,"['8412c99ffffffff', '8412c9dffffffff', '8412c9...","['8412c95ffffffff', '8412c97ffffffff', '8412c9..."
2,8412c93ffffffff,48.770008,-111.613230,48.770096,-111.862615,18277,8412c93ffffffff,True,"['8412c9bffffffff', '8412c91ffffffff', '8412c9...","['8412c9bffffffff', '8412c91ffffffff', '8412c9..."
3,8412c95ffffffff,48.983686,-112.624342,48.960892,-112.753612,9770,8412c95ffffffff,True,"['8412c9dffffffff', '8412c83ffffffff', '8412cb...","['8412cb9ffffffff', '8412cbbffffffff', '8412c9..."
4,8412c97ffffffff,48.689861,-112.187765,48.734037,-112.204917,5071,8412c97ffffffff,True,"['8412c91ffffffff', '8412c95ffffffff', '8412cb...","['8412c91ffffffff', '8412c95ffffffff', '8412cb..."
...,...,...,...,...,...,...,...,...,...,...
4528,8448f6dffffffff,29.188448,-104.068862,29.321720,-104.008415,15937,8448f6dffffffff,True,"['8448a97ffffffff', '8448abbffffffff', '8448ab...","['8448ab3ffffffff', '8448f65ffffffff', '8448f6..."
4529,8428d11ffffffff,48.501888,-122.894393,48.503282,-122.894348,155,8428d11ffffffff,True,"['8428d19ffffffff', '8428d1dffffffff', '8428d1...","['8428d19ffffffff', '8428d1dffffffff', '8428d1..."
4530,8427451ffffffff,45.624125,-85.456593,45.659515,-85.503197,5349,8427451ffffffff,True,"['8427459ffffffff', '842745dffffffff', '842745...","['8427459ffffffff', '842745dffffffff', '842745..."
4531,8427701ffffffff,48.102192,-88.868605,48.055263,-89.540355,50176,8427705ffffffff,False,"['8427709ffffffff', '842770dffffffff', '842770...","['8427705ffffffff', '8427707ffffffff']"


In [ ]:

def get_road_dist(origin: tuple[float, float], destination: tuple[float, float]) -> float:
    # origin/destination = (lat, lon)

    # meters → miles
    x = 0.000621371

    query = text("""
    WITH
    start_vertex AS (
        SELECT id
        FROM ways_vertices_pgr
        ORDER BY the_geom <-> ST_SetSRID(ST_Point(:origin_lon, :origin_lat), 4326)
        LIMIT 1
    ),
    end_vertex AS (
        SELECT id
        FROM ways_vertices_pgr
        ORDER BY the_geom <-> ST_SetSRID(ST_Point(:destination_lon, :destination_lat), 4326)
        LIMIT 1
    ),
    route AS (
        SELECT *
        FROM pgr_dijkstra(
            'SELECT id, source, target, cost, reverse_cost FROM ways',
            (SELECT id FROM start_vertex),
            (SELECT id FROM end_vertex),
            directed := false
        )
    )
    SELECT SUM(cost) AS distance_meters
    FROM route;
    """)

    with ENGINE.begin() as conn:
        df = pd.read_sql(
            query,
            conn,
            params={
                "origin_lat": origin[0],
                "origin_lon": origin[1],
                "destination_lat": destination[0],
                "destination_lon": destination[1],
            }
        )

    distance = df.iloc[0]["distance_meters"] * x
    return distance

def get_transit_to_valid_neighbors(df: pd.DataFrame) -> pd.DataFrame:
    
    # For each CellID
    # 1 check for ValidSnap=True
    # 2 For ValidNeighbors (list)
    #   1 look up RSLatitude & RSLongitude
    #   2 Use Database to find 
    return df